In [1]:
%pip install scipy==1.8.0 numpy==1.21.5
from rosbag2_reader_py import Rosbag2Reader

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Open ROS 2 Bag

**The file `rosbag2_reader_py.py` shall be in the same folder of the notebook.**

In [7]:
path = "./rosbag2_2026_11_27-11_57_05"

reader = Rosbag2Reader(path)
topics = reader.all_topics
topics

[INFO] [1795777746.678665116] [rosbag2_storage]: Opened database './rosbag2_2026_11_27-11_57_05/rosbag2_2026_11_27-11_57_05_0.db3' for READ_ONLY.
[INFO] [1795777746.679158588] [rosbag2_storage]: Opened database './rosbag2_2026_11_27-11_57_05/rosbag2_2026_11_27-11_57_05_0.db3' for READ_ONLY.


{'/ekf': 'nav_msgs/msg/Odometry',
 '/ekf_ext': 'nav_msgs/msg/Odometry',
 '/odom': 'nav_msgs/msg/Odometry',
 '/ground_truth': 'nav_msgs/msg/Odometry',
 '/imu': 'sensor_msgs/msg/Imu',
 '/landmarks': 'landmark_msgs/msg/LandmarkArray'}

## Get total number of messages in the bag

In [3]:
tot_msgs = 0
for _ in reader:
    tot_msgs += 1

print(f"Total messages: {tot_msgs}")

[INFO] [1795777609.996809480] [rosbag2_storage]: Opened database './rosbag2_2026_11_27-11_57_05/rosbag2_2026_11_27-11_57_05_0.db3' for READ_ONLY.


Total messages: 54905


## Select messages of specific topics

In [29]:
tot_msgs = 0
reader.set_filter(["/ekf_ext"])
for _ in reader:
    tot_msgs += 1

print("After the filter is applyed: ", reader.selected_topics)
print(f"Total messages: {tot_msgs}")

reader.reset_filter() # if you want to read all messages after you set a filter
print("After the filter is reset: ", reader.selected_topics)

[INFO] [1795778791.970766850] [rosbag2_storage]: Opened database './rosbag2_2026_11_27-11_57_05/rosbag2_2026_11_27-11_57_05_0.db3' for READ_ONLY.
[INFO] [1795778791.971669153] [rosbag2_storage]: Opened database './rosbag2_2026_11_27-11_57_05/rosbag2_2026_11_27-11_57_05_0.db3' for READ_ONLY.


After the filter is applyed:  {'/ekf_ext': 'nav_msgs/msg/Odometry'}
Total messages: 26105
After the filter is reset:  {'/ekf': 'nav_msgs/msg/Odometry', '/ekf_ext': 'nav_msgs/msg/Odometry', '/odom': 'nav_msgs/msg/Odometry', '/ground_truth': 'nav_msgs/msg/Odometry', '/imu': 'sensor_msgs/msg/Imu', '/landmarks': 'landmark_msgs/msg/LandmarkArray'}


[INFO] [1795778793.929463462] [rosbag2_storage]: Opened database './rosbag2_2026_11_27-11_57_05/rosbag2_2026_11_27-11_57_05_0.db3' for READ_ONLY.


## Access only data of a given type

In this example you can see  how to access an `Odometry` message checking for its type.

Please, notice the difference between **recording time** and time reported in the **stamp**. This is because the message was generated at a time that does not coincide with the time the message was received and recorded. This difference in a real robot may be really small, in the order of microseconds, but for a simulation, as in the reported case, the time could be extremely different. **You shall always use `header.stamp` whenever it is available.**

In [16]:
from rclpy.time import Time
from nav_msgs.msg import Odometry
for topic_name, msg, t in reader:
    print(f"Received message of type {type(msg).__name__} on topic {topic_name} recorded at time {t}")
    if type(msg) is Odometry:
        time = Time.from_msg(msg.header.stamp).nanoseconds
        print(f"Position (x, y) at time {time}: ({msg.pose.pose.position.x:.2f}, {msg.pose.pose.position.y:.2f})")
        break

Received message of type Odometry on topic /ground_truth recorded at time 1795777025082119885
Position (x, y) at time 24649000000: (-2.00, -0.50)


[INFO] [1795777906.822233683] [rosbag2_storage]: Opened database './rosbag2_2026_11_27-11_57_05/rosbag2_2026_11_27-11_57_05_0.db3' for READ_ONLY.


## Interpolate data to compute metrics

In order to compute the metrics for `/odom` and `/ekf` topics, you have to compare the poses reported in these topic with the poses reported in topic `/ground_truth` in the same time instants. 

Since the data are generated from different nodes at different frequencies, the time of the various topics will be different. So, we need to interpolate ground truth data on the time scale of the topic we want to evaluate.

First of all, let us save relevant data from messages in some NumPy arrays. As you can see from the output, the number of points from the two topics is different.

In [28]:
import numpy as np
from scipy.interpolate import interp1d

time_gt = []
gt_data = []
time_ekf = []
ekf_data = []
time_ekf_ext = []
ekf_ext_data = []

reader.set_filter(["/ground_truth", "/ekf","/ekf_ext"])

for topic_name, msg, t in reader:
    if topic_name == "/ground_truth":
        time_gt.append(Time.from_msg(msg.header.stamp).nanoseconds)
        gt_data.append((msg.pose.pose.position.x, msg.pose.pose.position.y))
    elif topic_name == "/ekf":
        time_ekf.append(Time.from_msg(msg.header.stamp).nanoseconds)
        ekf_data.append((msg.pose.pose.position.x, msg.pose.pose.position.y))
    elif topic_name == "/ekf_ext":
        time_ekf_ext.append(Time.from_msg(msg.header.stamp).nanoseconds)
        ekf_ext_data.append((msg.pose.pose.position.x, msg.pose.pose.position.y))

time_gt = np.array(time_gt)
gt_data = np.array(gt_data)
time_ekf = np.array(time_ekf)
ekf_data = np.array(ekf_data)
time_ekf_ext = np.array(time_ekf_ext)
ekf_ext_data = np.array(ekf_ext_data)

print(f"Ground truth points: {len(gt_data)}")
print(f"EKF points: {len(ekf_data)}")
print(f"Extended-EKF points: {len(ekf_ext_data)}")

[INFO] [1795778759.548138341] [rosbag2_storage]: Opened database './rosbag2_2026_11_27-11_57_05/rosbag2_2026_11_27-11_57_05_0.db3' for READ_ONLY.
[INFO] [1795778759.548879491] [rosbag2_storage]: Opened database './rosbag2_2026_11_27-11_57_05/rosbag2_2026_11_27-11_57_05_0.db3' for READ_ONLY.


Ground truth points: 2073
EKF points: 2434
Extended-EKF points: 26105


Now, let us create an interpolating function using SciPy `interp1d`.

In [30]:
gt_interpol = interp1d(time_gt, gt_data, axis=0, fill_value="extrapolate", kind="nearest")
gt_data_interp = gt_interpol(time_ekf)
gt_ext_data_interp = gt_interpol(time_ekf_ext)

print(f"Interpolated ground truth points for EKF: {len(gt_data_interp)}")
print(f"Interpolated ground truth points for Extended-EKF: {len(gt_ext_data_interp)}")

Interpolated ground truth points for EKF: 2434
Interpolated ground truth points for Extended-EKF: 26105


Compute Mean Absolute Error between odometry data and interpolated ground truth. You can find already implemented metrics functionson Portale della Didattica (Lecture_notebooks/Gaussian_filters.zip/utils.py)

## Metrics

In [31]:
def _error(actual: np.ndarray, predicted: np.ndarray):
    """ Simple error """
    return actual - predicted

def mse(actual: np.ndarray, predicted: np.ndarray):
    """ Mean Squared Error """
    if len(actual.shape)==1 and len(predicted.shape)==1:
        return np.mean(np.square(_error(actual, predicted)), axis=0)
    return np.mean(np.sum(np.square(_error(actual, predicted)), axis=1), axis=0)

def rmse(actual: np.ndarray, predicted: np.ndarray):
    """ Root Mean Squared Error """
    return np.sqrt(mse(actual, predicted))

def mae(error: np.ndarray):
    """ Mean Absolute Error """
    return np.mean(np.abs(error))

In [ ]:
# Mean Squared Error
mse_ekf=mse(gt_data_interp, ekf_data)
# Mean Absolute Error
mae_ekf=np.mean(np.linalg.norm(ekf_data - gt_data_interp, axis=1), axis=0)
# Root Mean Square Error
rmse_ekf=rmse(gt_data_interp, ekf_data)

In [32]:
# Mean Squared Error
mse_ekf_ext=mse(gt_ext_data_interp, ekf_ext_data)
# Mean Absolute Error
mae_ekf_ext=np.mean(np.linalg.norm(ekf_ext_data - gt_ext_data_interp, axis=1), axis=0)
# Root Mean Square Error
rmse_ekf_ext=rmse(gt_ext_data_interp, ekf_ext_data)

In [24]:
print("mse_ekf: ",mse_ekf)
print("mae_ekf: ",mae_ekf)
print("rmse_ekf: ",rmse_ekf)

mse_ekf:  0.02830066701230335
mae_ekf:  0.06750490100012733
rmse_ekf:  0.16822802088921854


In [33]:
print("mse_ekf_ext: ",mse_ekf_ext)
print("mae_ekf_ext: ",mae_ekf_ext)
print("rmse_ekf_ext: ",rmse_ekf_ext)

mse_ekf_ext:  0.030512756770430603
mae_ekf_ext:  0.06509847323886311
rmse_ekf_ext:  0.17467901067509686
